In [ ]:
# ============================================================
# 第 5 章 Part-1：为「有监督指令微调」(supervised instruction finetuning) 准备数据集
# ------------------------------------------------------------
# 本 notebook 的目标：理解 Alpaca 风格的指令数据格式，并实现 format_input
# 把 instruction / input / output 三段式数据拼接成统一的提示模板 (prompt template)。
# ============================================================
#Preparing a dataset for supervised instruction finetuning

In [ ]:
# 目的：加载指令微调数据集（JSON 文件）。
# 该数据集每条是一个 dict，包含 instruction（指令）、input（可选的输入）、output（期望输出）三个字段，
# 这就是所谓的 Alpaca 风格指令数据（源自 Stanford Alpaca 项目）。
import json


# 数据文件路径：来自配套仓库 LLM-workshop-2024 的第 6 章目录
file_path = "LLM-workshop-2024/06_finetuning/instruction-data.json"

# 以只读方式打开并用 json.load 反序列化为 Python 列表（list of dict）
with open(file_path, "r") as file:
    data = json.load(file)
# 打印样本总数，确认数据加载成功
print("Number of entries:", len(data))

In [ ]:
# 查看第 50 条样本，观察 Alpaca 三段式结构：instruction / input / output。
# 注意：这一条通常「有 input」（例如给定一个句子，要求改写/分类），
# 用来对比下面那条「没有 input」的情况。
print("Example entry:\n", data[50])

In [ ]:
# 查看第 999 条样本。这一条一般是「input 为空」的情形，
# 只有 instruction 和 output（例如「说出某国首都」这类无需额外输入的任务）。
print("Another example entry:\n", data[999])

In [ ]:
# 核心函数 format_input：把一条样本拼接成统一的提示模板 (prompt template)。
# 为什么要统一模板？指令微调时必须让模型在「固定格式」下学习「看到指令 -> 产生回答」的映射；
# 推理时也用同样的模板包裹输入，训练与推理格式一致，模型才能稳定地识别指令边界。
# 本模板采用经典 Alpaca 风格：前言 + ### Instruction + （可选）### Input，
# 而 ### Response 部分在训练时由 output 补上、推理时留空让模型续写。
def format_input(entry):
    # 固定前言（system-style preamble）+ 指令段落。前言告诉模型「下面是一个任务，请恰当完成」。
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # 仅当 input 字段非空时才追加 ### Input 段落；为空则用空字符串占位（三元表达式）。
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    # 返回「前言+指令(+输入)」拼接结果，不含 Response 段落。
    return instruction_text + input_text

In [ ]:
# 演示：对「有 input」的第 50 条构造完整提示（含期望回答）。
model_input = format_input(data[50])
# 手动补上 ### Response 段落 —— 训练时这部分就是模型要学习生成的目标 (label)。
desired_response = f"\n\n### Response:\n{data[50]['output']}"

# 打印拼出的完整训练样本文本：前言 + Instruction + Input + Response。
print(model_input + desired_response)

In [ ]:
# 演示：对「无 input」的第 999 条构造完整提示。
# 对比上一格可见：input 为空时模板里不会出现 ### Input 段落，模型只依据指令作答。
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)